# Opener vs OWNER baselines — dashboard AMI par dataset

Line chart interactif des scores AMI par dataset, avec widget pour sélectionner les modèles et filtrer par catégorie de setup. Données : Table 1 du papier OWNER (IEEE Access 2025) + nos résultats Opener (classifier sweep).

## Setting d'Opener (important)

Opener est **Zero-Shot supervised cross-domain** :
- Il voit la liste des labels via YAML → ≠ unsupervised pur.
- Le fit utilise les spans gold du train → supervised.
- Embedder + MD figés (pas de fine-tuning sur la cible) → transfer cross-domain.

→ La comparaison **équitable** se fait avec les baselines **Zero-Shot** du papier (UniNER, GliNER L, GNER, GoLLIE, ChatIE). Les baselines **Unsupervised & Open-World** (OWNER, ChatIE Uns, UniNER Uns) ne voient PAS la liste des labels → comparaison biaisée, à afficher seulement comme contexte de référence.

## Datasets affichés

Conservés (que nous avons benchmarkés) : **5 CrossNER + FabNER + MIT-Restaurant + WNUT 17** = 8 datasets.

Retirés : GENIA, i2b2, GENTLE, GUM (licence/non Parquet), MIT-Movie (mapping label incertain).

Note : nos résultats Opener sur CrossNER sont **agrégés sur les 5 sous-domaines** (notre mirror HF ne sépare pas le sub-domain). On affiche donc la même valeur sur les 5 colonnes CrossNER pour Opener, signalée par `(agg)`.

In [1]:
# === Données AMI (%) — Table 1 du papier OWNER + nos résultats Opener ===
DATASETS = [
    'CrossNER-AI', 'CrossNER-Literature', 'CrossNER-Music',
    'CrossNER-Politics', 'CrossNER-Science',
    'FabNER', 'MIT-Restaurant', 'WNUT 17',
]

# Nos résultats Opener (classifier sweep 2026-05-29, Nomic 768 + classifieurs sklearn)
# CrossNER chez nous = score agrégé sur les 5 sous-domaines mélangés
OUR = {
    'Opener-GMM (agg)':    {'crossner_agg': 37.4, 'FabNER':  8.0, 'MIT-Restaurant': 14.5, 'WNUT 17': 14.2},
    'Opener-LogReg (agg)': {'crossner_agg': 36.4, 'FabNER':  8.7, 'MIT-Restaurant': 18.5, 'WNUT 17': 17.3},
    'Opener-SVM (agg)':    {'crossner_agg': 39.6, 'FabNER':  9.9, 'MIT-Restaurant': 29.1, 'WNUT 17': 16.0},
}

# Baselines zero-shot (OWNER paper Table 1)
ZS = {
    'UniNER':        [43.1, 48.6, 50.2, 46.6, 49.4, 23.5, 23.8, 24.2],
    'GoLLIE':        [48.0, 50.2, 52.6, 49.7, 52.7, 21.1, 29.4, 32.3],
    'GliNER L':      [45.1, 50.7, 58.4, 50.0, 54.1, 27.9, 37.1, 30.3],
    'ChatIE':        [39.4, 46.5, 49.8, 43.0, 52.1, 25.2, 33.8, 24.1],
    'GNER (T5)':     [41.9, 45.9, 52.7, 49.1, 50.2, 18.3, 35.9, 28.8],
    'GNER (T5-xxl)': [52.5, 53.7, 63.1, 54.9, 59.7, 14.7, 42.1, 31.0],
}

# Baselines unsupervised & open-world (OWNER paper Table 1)
UN = {
    'ChatIE Uns':       [12.8, 20.4, 27.8, 27.5, 25.7,  5.7, 17.7,  6.8],
    'UniNER Uns':       [33.5, 42.8, 48.1, 40.3, 43.7, 21.2, 29.0, 15.0],
    'OWNER (CoNLL)':    [44.3, 46.6, 50.1, 53.7, 52.3, 14.7, 12.1, 24.6],
    'OWNER (Pile-NER)': [39.4, 49.5, 52.5, 48.5, 50.9, 23.5, 27.9, 24.0],
}

# Reconstruit le tableau AMI[dataset][model]
AMI = {ds: {} for ds in DATASETS}
for fam in (ZS, UN):
    for m, vals in fam.items():
        for ds, v in zip(DATASETS, vals):
            AMI[ds][m] = v
# Opener : agg constant sur les 5 CrossNER
for m, vs in OUR.items():
    for ds in DATASETS:
        if ds.startswith('CrossNER'):
            AMI[ds][m] = vs['crossner_agg']
        else:
            AMI[ds][m] = vs.get(ds)

CATEGORY = {**{m: 'Zero-Shot' for m in ZS},
            **{m: 'Unsupervised & Open-World' for m in UN},
            **{m: 'Zero-Shot (ours)' for m in OUR}}

ALL_MODELS = list(ZS) + list(UN) + list(OUR)
print(f'{len(ALL_MODELS)} modèles · {len(DATASETS)} datasets')

13 modèles · 8 datasets


In [5]:
# === Widget interactif ===
%matplotlib inline
import matplotlib.pyplot as plt
from ipywidgets import interact, SelectMultiple, Dropdown, Checkbox, Layout

DEFAULTS = ['GliNER L', 'GNER (T5-xxl)', 'OWNER (Pile-NER)', 'Opener-GMM (agg)', 'Opener-SVM (agg)']
PALETTE = plt.cm.tab20.colors

def plot(models, category, highlight_opener):
    fig, ax = plt.subplots(figsize=(13, 6))
    plotted = 0
    for i, m in enumerate(models):
        cat = CATEGORY.get(m, '?')
        if category != 'All' and not (category == cat or (category == 'Zero-Shot' and cat == 'Zero-Shot (ours)')):
            continue
        y = [AMI[ds].get(m) for ds in DATASETS]
        is_ours = m.startswith('Opener')
        lw = 3.0 if (highlight_opener and is_ours) else 1.6
        ls = '-' if is_ours else '--'
        marker = 'o' if is_ours else 's'
        ax.plot(range(len(DATASETS)), y, marker=marker, label=m,
                color=PALETTE[ALL_MODELS.index(m) % len(PALETTE)],
                linewidth=lw, linestyle=ls, markersize=7 if is_ours else 5)
        plotted += 1
    ax.set_xticks(range(len(DATASETS)))
    ax.set_xticklabels(DATASETS, rotation=30, ha='right')
    ax.set_xlabel('Dataset')
    ax.set_ylabel('AMI (%)')
    ax.set_title(f'AMI by dataset  ·  filter = {category}  ·  {plotted} courbes affichées')
    ax.grid(alpha=0.3)
    if plotted:
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

interact(
    plot,
    models=SelectMultiple(options=ALL_MODELS, value=DEFAULTS, description='Modèles',
                          layout=Layout(width='450px', height='240px')),
    category=Dropdown(options=['All', 'Zero-Shot', 'Unsupervised & Open-World'],
                      value='All', description='Catégorie'),
    highlight_opener=Checkbox(value=True, description='Mettre Opener en avant'),
);

interactive(children=(SelectMultiple(description='Modèles', index=(2, 5, 9, 10, 12), layout=Layout(height='240…

## Lecture

- **Opener** (lignes pleines, points ronds) : nos 3 classifieurs comparés directement aux baselines.
- **Baselines papier** (lignes pointillées, points carrés) : du papier OWNER (IEEE Access 2025).
- **Catégorie « Zero-Shot »** = comparaison équitable avec Opener (voient les noms des labels).
- **Catégorie « Unsupervised & Open-World »** = baselines qui découvrent les types sans connaître la liste → contexte, comparaison non-équivalente avec Opener.

### Observations attendues

- Sur les 5 CrossNER : Opener-SVM (~39.6 agg) est nettement sous les zero-shot baselines (UniNER ~46, GliNER L ~52). Le gap reflète l'absence de fine-tuning de l'embedding.
- Sur FabNER : tous les modèles plafonnent bas (~8-28). FabNER reste un trou de vocabulaire pour les embedders web text.
- Sur MIT-Restaurant : Opener-SVM (29.1) est compétitif avec UniNER (23.8) — joli résultat low-compute.
- Sur WNUT 17 : Opener-LogReg/SVM (16-17) reste sous les baselines (~24-37) → setting bruité dur.